In [1]:
!uv pip install vllm==0.10.1 --torch-backend=auto

Using Python 3.12.11 environment at: /usr
Resolved 140 packages in 2.03s
Prepared 55 packages in 38.58s
Uninstalled 21 packages in 817ms
Installed 55 packages in 243ms
 + astor==0.8.1
 + blake3==1.0.6
 + cbor2==5.7.0
 + compressed-tensors==0.10.2
 + depyf==0.19.0
 + diskcache==5.6.3
 + dnspython==2.8.0
 + email-validator==2.3.0
 + fastapi-cli==0.0.13
 + fastapi-cloud-cli==0.2.1
 + gguf==0.17.1
 + httptools==0.6.4
 + interegular==0.3.3
 + llguidance==0.7.30
 - llvmlite==0.43.0
 + llvmlite==0.44.0
 + lm-format-enforcer==0.10.12
 + mistral-common==1.8.5
 + msgspec==0.19.0
 + ninja==1.13.0
 - numba==0.60.0
 + numba==0.61.2
 - nvidia-cublas-cu12==12.6.4.1
 + nvidia-cublas-cu12==12.8.3.14
 - nvidia-cuda-cupti-cu12==12.6.80
 + nvidia-cuda-cupti-cu12==12.8.57
 - nvidia-cuda-nvrtc-cu12==12.6.77
 + nvidia-cuda-nvrtc-cu12==12.8.61
 - nvidia-cuda-runtime-cu12==12.6.77
 + nvidia-cuda-runtime-cu12==12.8.57
 - nvidia-cudnn-cu12==9.10.2.21
 + nvidia-cudnn-cu12==9.7.1.26
 - nvidia-cufft-cu12==11.3.0.4


In [ ]:
!huggingface-cli login

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
The token `testvllm` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.

In [ ]:
!vllm serve meta-llama/Llama-3.2-11B-Vision-Instruct \
  --enforce-eager --max-num-seqs 1 --max-model-len 4096 \
  --gpu-memory-utilization 0.90

2025-08-29 10:06:40.090288: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-29 10:06:40.109165: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756462000.131306    1404 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756462000.137810    1404 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1756462000.154867    1404 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
# !vllm serve meta-llama/Llama-3.1-8B-Instruct --gpu-memory-utilization 0.90
!vllm serve meta-llama/Llama-3.2-11B-Vision-Instruct --enforce-eager --max-num-seqs 16 --gpu-memory-utilization 0.90

In [3]:
# !nohup vllm serve meta-llama/Llama-3.2-11B-Vision-Instruct \
#   --enforce-eager --max-num-seqs 1 --max-model-len 4096 \
#   --gpu-memory-utilization 0.90 \
#   > vllm.log 2>&1 &
# !sleep 3 && tail -n 40 vllm.log
!nohup vllm serve openai/gpt-oss-20b \
  --host 0.0.0.0 --port 8000 \
  --max-model-len 131072 --gpu-memory-utilization 0.90 \
  > vllm.log 2>&1 &
!sleep 3 && tail -n 40 vllm.log

In [4]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# start tunnel in background; logs -> cfd.log
!nohup ./cloudflared tunnel --url http://127.0.0.1:8000 > cfd.log 2>&1 &
!sleep 3 && grep -o 'https://[-a-z0-9\.]*trycloudflare.com' -m1 cfd.log

In [5]:
!grep -o 'https://[-a-z0-9\.]*trycloudflare.com' -m1 cfd.log

https://option-suites-louisiana-apps.trycloudflare.com


In [6]:
from openai import OpenAI

# Point the client to your local vLLM server
client = OpenAI(base_url="https://option-suites-louisiana-apps.trycloudflare.com/v1", api_key="EMPTY")

response = client.chat.completions.create(
    # model="meta-llama/Llama-3.2-11B-Vision-Instruct",
    model = "openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": "translate this to vietnamese: Dyshidrotic eczema (pompholyx)"}
    ],
    # max_tokens=200
)

print(response.choices[0].message.content)


Viêm da dyshidrotic (pompholyx)


In [ ]:
response.choices[0]

Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning_content='User wants translation to Vietnamese: "Dyshidrotic eczema (pompholyx)". This is presumably "eczema dyshidrotic (pompholyx)" but we need Vietnamese translation. Let\'s think: "Dyshidrotic eczema" is a specific type of eczema characterized by vesicles on the palms/soles. Vietnamese translation: "Éczema buột tay" (common term?). Might translate to "Eczema dạng nắm tay" or "Éczema dyshidrotic" rarely. Common phrase: "Eczema nắm tay (dolam)". Also "pompholyx" is the same. So likely translation: "Éczema nắn tay (Dyshidrotic)". Might prefer "Éczema Lí Vảy"?? Let\'s see: In Vietnamese dermatology, "Eczema chuyển động" no. Actually "Dyshidrotic eczema" is called "Eczema nắm tay" or "'), stop_reason=None)

In [2]:
!vllm serve openai/gpt-oss-20b

2025-09-27 05:23:17.934056: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-27 05:23:17.954360: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758950597.977957    3810 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758950597.985917    3810 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1758950598.005876    3810 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 